In [2]:
import os
import sys
import subprocess

sys.path.insert(0, '..')
from src import *

# Strains

## Velocity const

In [14]:
save_every = 2
output_dir = Filepaths.DATA.value / 'strains' / f'gpu_dt0.002'
os.makedirs(output_dir, exist_ok=True)
print(output_dir)

/home/fgarbuzov/Documents/lammps_test/data/strains/gpu_dt0.002


In [19]:
script_gen = ScriptGenLJ(output_dir)
script = script_gen.init_box_atoms()
script += script_gen.computes(comp_born=False)
script += script_gen.time_step_log(time_step=0.002, thermo_log_steps=10000)
script += script_gen.run_equilibration(10000)
script += script_gen.save_fixes({FileKeys.THERMO.value: save_every,
                                 FileKeys.PRESS.value: save_every}, append=False)
script += "\nfix nve_prod all nve\n"

In [20]:
script += """
variable Lx0 equal lx

variable i loop 10
label loop_start

print "Cycle ${i}"
run 10000
change_box all x scale 1.001 remap
run 10000
change_box all x final 0 ${Lx0} remap units box

next i
jump SELF loop_start
"""

In [21]:
print(script)

# Initialization
units lj
dimension 3
boundary p p p
atom_style atomic

# Create simulation box and atoms
lattice fcc 1.058 origin 0.05 0.05 0.05
region simbox block 0 10 0 10 0 10 units lattice
create_box 1 simbox
create_atoms 1 box
mass 1 1.0
velocity all create 4.0 94673 mom yes

# Potential settings
pair_style lj/smooth 2.5 3.5
pair_coeff 1 1 1.0 1.0
pair_modify shift yes

# Monitoring variables
variable time equal time
variable etot equal etotal
variable vol equal vol
variable dens equal density

# Default computes
compute temp all temp
compute press all pressure thermo_temp

# Timestep (default 0.005 for lennard-jones)
timestep 0.002

# Screen and log output
thermo 10000
thermo_style custom step temp press etotal
thermo_modify flush yes

fix nvt_eq all nvt temp 4.0 4.0 0.2
run 10000
unfix nvt_eq

# Output values
fix thermo_output all ave/time 1 1 2 v_time c_thermo_temp v_etot v_vol v_dens &
    file "/home/fgarbuzov/Documents/lammps_test/data/strains/gpu_dt0.002/thermo.dat"
fix p

In [22]:
script_name = 'strains_N4000_dt0.002.in'
with open(Filepaths.INPUTS.value / script_name, 'w') as f:
    f.write(script)

## Velocity remap

# Convergence

## First script

In [12]:
save_every = 4
output_dir = Filepaths.DATA.value / 'convergence' / f'gpu_dt0.005'
os.makedirs(output_dir, exist_ok=True)
print(output_dir)

/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005


In [13]:
script_gen = ScriptGenLJ(output_dir)
script = script_gen.init_box_atoms()
script += script_gen.computes()
script += script_gen.time_step_log(thermo_log_steps=10000)
script += script_gen.run_equilibration(10000)
script += script_gen.save_fixes({FileKeys.THERMO.value: save_every,
                                 FileKeys.PRESS.value: save_every,
                                 FileKeys.PRESS_POT.value: save_every,
                                 FileKeys.BORN.value: save_every}, append=False)
script += script_gen.run_production(10_000_000)
script += script_gen.write('snapshot_10M')

In [14]:
print(script)

# Initialization
units lj
dimension 3
boundary p p p
atom_style atomic

# Create simulation box and atoms
lattice fcc 1.058 origin 0.05 0.05 0.05
region simbox block 0 10 0 10 0 10 units lattice
create_box 1 simbox
create_atoms 1 box
mass 1 1.0
velocity all create 4.0 94673 mom yes

# Potential settings
pair_style lj/smooth 2.5 3.5
pair_coeff 1 1 1.0 1.0
pair_modify shift yes

# Monitoring variables
variable time equal time
variable etot equal etotal
variable vol equal vol
variable dens equal density

# Default computes
compute temp all temp
compute press all pressure thermo_temp

# Born matrix computation
compute press_pot all pressure NULL virial
compute born_matrix all born/matrix numdiff 1e-06 press_pot

# Timestep (default 0.005 for lennard-jones)
timestep 0.005

# Screen and log output
thermo 10000
thermo_style custom step temp press etotal
thermo_modify flush yes

fix nvt_eq all nvt temp 4.0 4.0 0.5
run 10000
unfix nvt_eq

# Output values
fix thermo_output all ave/time 1 1 4 v_t

In [15]:
script_name = 'N4000_dt0.005.in'
with open(Filepaths.INPUTS.value / script_name, 'w') as f:
    f.write(script)

In [16]:
output_dir

PosixPath('/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005')

In [ ]:
subprocess.run(f"lmp -sf gpu -pk gpu 1 -i {Filepaths.INPUTS.value / script_name} -nocite \
                -log {output_dir / Filenames.SIM_LOG.value}", shell=True)

LAMMPS (22 Jul 2025 - Update 4)
  using 12 OpenMP thread(s) per MPI task
Lattice spacing in x,y,z = 1.5578469 1.5578469 1.5578469
Created orthogonal box = (0 0 0) to (15.578469 15.578469 15.578469)
  1 by 1 by 1 MPI processor grid
Created 4000 atoms
  using lattice units in orthogonal box = (0 0 0) to (15.578469 15.578469 15.578469)
  create_atoms CPU = 0.001 seconds

--------------------------------------------------------------------------
- Using acceleration for lj/smooth:
-  with 1 proc(s) per device.
-  with 12 thread(s) per proc.
-  Horizontal vector operations: ENABLED
-  Shared memory system: No
--------------------------------------------------------------------------
Device 0: NVIDIA GeForce RTX 3060, 28 CUs, 10/12 GB, 1.8 GHZ (Mixed Precision)
--------------------------------------------------------------------------

Initializing Device and compiling on process 0...Done.
Initializing Device 0 on core 0...Done.

Generated 0 of 0 mixed pair_coeff terms from geometric mixing 

## Continue

In [33]:
save_every = 4
output_dir = Filepaths.DATA.value / 'convergence' / f'gpu_dt0.005'
os.makedirs(output_dir, exist_ok=True)
print(output_dir)

/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005


In [34]:
script_gen = ScriptGenLJ(output_dir)
script = script_gen.read('snapshot_25M')[1:]
script += script_gen.computes()
script += script_gen.time_step_log(thermo_log_steps=10000)
script += script_gen.save_fixes({FileKeys.THERMO.value: save_every,
                                 FileKeys.PRESS.value: save_every,
                                 FileKeys.PRESS_POT.value: save_every,
                                 FileKeys.BORN.value: save_every}, append=True)
script += script_gen.run_production(5_000_000)
script += script_gen.write('snapshot_30M')
#script += script_gen.run_production(5_000_000, fix=False)
#script += script_gen.write('snapshot_25M')

In [35]:
print(script)

read_restart "/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005/snapshot_25M.restart"

# Monitoring variables
variable time equal time
variable etot equal etotal
variable vol equal vol
variable dens equal density

# Default computes
compute temp all temp
compute press all pressure thermo_temp

# Born matrix computation
compute press_pot all pressure NULL virial
compute born_matrix all born/matrix numdiff 1e-06 press_pot

# Timestep (default 0.005 for lennard-jones)
timestep 0.005

# Screen and log output
thermo 10000
thermo_style custom step temp press etotal
thermo_modify flush yes

# Output values
fix thermo_output all ave/time 1 1 4 v_time c_thermo_temp v_etot v_vol v_dens &
    append "/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005/thermo.dat"
fix press_output all ave/time 1 1 4 v_time c_press[*] &
    append "/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005/press.dat"
fix press_pot_output all ave/time 1 1 4 v_time c_press_pot[

In [36]:
script_name = 'N4000_dt0.005_continue.in'
with open(Filepaths.INPUTS.value / script_name, 'w') as f:
    f.write(script)

In [37]:
output_dir

PosixPath('/home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005')

In [38]:
subprocess.run(f"lmp -sf gpu -pk gpu 1 -i {Filepaths.INPUTS.value / script_name} -nocite \
                -log {output_dir / Filenames.SIM_LOG.value}", shell=True)

LAMMPS (22 Jul 2025 - Update 4)
  using 12 OpenMP thread(s) per MPI task
Reading restart file ...
  restart file = 22 Jul 2025, LAMMPS = 22 Jul 2025
  restoring atom style atomic from restart
  orthogonal box = (0 0 0) to (15.578469 15.578469 15.578469)
  1 by 1 by 1 MPI processor grid
  restoring pair style lj/smooth/gpu from restart
  4000 atoms
  read_restart CPU = 0.002 seconds

--------------------------------------------------------------------------
- Using acceleration for lj/smooth:
-  with 1 proc(s) per device.
-  with 12 thread(s) per proc.
-  Horizontal vector operations: ENABLED
-  Shared memory system: No
--------------------------------------------------------------------------
Device 0: NVIDIA GeForce RTX 3060, 28 CUs, 10/12 GB, 1.8 GHZ (Mixed Precision)
--------------------------------------------------------------------------

Initializing Device and compiling on process 0...Done.
Initializing Device 0 on core 0...Done.

Generated 0 of 0 mixed pair_coeff terms from ge

CompletedProcess(args='lmp -sf gpu -pk gpu 1 -i /home/fgarbuzov/Documents/lammps_test/input_scripts/N4000_dt0.005_continue.in -nocite                 -log /home/fgarbuzov/Documents/lammps_test/data/convergence/gpu_dt0.005/sim.log', returncode=0)

# Test save frequency

In [3]:
for save_every in [3,10,30,100]:
    output_dir = Filepaths.DATA.value / 'test' / f'save{save_every}_dt0.002'
    os.makedirs(output_dir, exist_ok=True)
    script_gen = ScriptGenLJ(output_dir)
    script = script_gen.init_box_atoms()
    script += script_gen.computes()
    script += script_gen.time_step_log(time_step=0.002)
    script += script_gen.run_equilibration(5000)
    script += script_gen.save_fixes({FileKeys.THERMO.value: save_every,
                                    FileKeys.PRESS.value: save_every,
                                    FileKeys.PRESS_POT.value: save_every,
                                    FileKeys.BORN.value: save_every}, append=False)
    script += script_gen.run_production(500000)
    with open(Filepaths.INPUTS.value / f'save{save_every}.test', 'w') as f:
        f.write(script)
    subprocess.run(f"lmp -sf gpu -pk gpu 1 -i {Filepaths.INPUTS.value / f'save{save_every}'}.test -nocite \
                   -log {output_dir / Filenames.SIM_LOG.value}", shell=True)

LAMMPS (22 Jul 2025 - Update 4)
  using 12 OpenMP thread(s) per MPI task
Lattice spacing in x,y,z = 1.5578469 1.5578469 1.5578469
Created orthogonal box = (0 0 0) to (15.578469 15.578469 15.578469)
  1 by 1 by 1 MPI processor grid
Created 4000 atoms
  using lattice units in orthogonal box = (0 0 0) to (15.578469 15.578469 15.578469)
  create_atoms CPU = 0.001 seconds

--------------------------------------------------------------------------
- Using acceleration for lj/smooth:
-  with 1 proc(s) per device.
-  with 12 thread(s) per proc.
-  Horizontal vector operations: ENABLED
-  Shared memory system: No
--------------------------------------------------------------------------
Device 0: NVIDIA GeForce RTX 3060, 28 CUs, 9.8/12 GB, 1.8 GHZ (Mixed Precision)
--------------------------------------------------------------------------

Initializing Device and compiling on process 0...Done.
Initializing Device 0 on core 0...Done.

Generated 0 of 0 mixed pair_coeff terms from geometric mixing